# Rayleigheva raspodjela: gađanje u metu strelicama pikada (knjiga, poglavlje 2.4.3)

$$p(x)=\frac{x}{\sigma^2}e^{-\frac12\left(\frac{x}{\sigma}\right)^2},\qquad
P(x)=1-e^{-\frac12\left(\frac{x}{\sigma}\right)^2},\qquad x\ge 0\qquad(2.14),(2.15)$$

**Eksperiment (primjer iz knjige).** Igrač gađa centar mete. Greška po horizontali $X$ i greška po vertikali $Y$ su
nezavisne Gaussove promjenljive $\mathcal N(0,\sigma^2)$ (ruka malo „drhti“ u oba smjera). Rayleigheva raspodjela opisuje
**udaljenost pogotka od centra**, $R=\sqrt{X^2+Y^2}$.

Vjerovatnoća da je pogodak unutar kruga poluprečnika $r$ računa se u polarnim koordinatama:

$$P(r)=\int_0^{2\pi}\!\!\int_0^r\frac{1}{2\pi\sigma^2}e^{-\frac{\rho^2}{2\sigma^2}}\rho\,d\rho\,d\theta
=1-e^{-\frac{r^2}{2\sigma^2}},\qquad p(r)=\frac{dP}{dr}=\frac{r}{\sigma^2}e^{-\frac{r^2}{2\sigma^2}} .$$

Isti rezultat se vidi i u Primjeru 2.15 iz knjige (Box–Mullerova transformacija, programski kodovi 2.44–2.47):
poluprečnik $\sqrt{-2\ln X}$ tačke iz dvodimenzionalne Gaussove raspodjele ima baš Rayleighevu raspodjelu.

**Pokretanje:** u Google Colabu `Runtime → Run all`. Prva ćelija sama klonira repozitorij
[ierovic1ba/ssp](https://github.com/ierovic1ba/ssp) (fork repozitorija [esokic/ssp](https://github.com/esokic/ssp) uz knjigu) i učita biblioteku `ssp`.
Svaka bilježnica radi $N=10\,000$ eksperimenata. Detaljna izvođenja su u PDF-u.

In [ ]:
# Priprema okruženja: kloniranje repozitorija sa SSP bibliotekom (prema README uputstvu)
import os, sys, subprocess

REPO_URL = "https://github.com/ierovic1ba/ssp.git"   # fork originala https://github.com/esokic/ssp
SSP_PUTANJA = "/content/ssp" if os.path.isdir("/content") else os.path.abspath("ssp")

if not os.path.isdir(os.path.join(SSP_PUTANJA, "ssp")):
    subprocess.run(["git", "clone", REPO_URL, SSP_PUTANJA], check=True)

sys.path.insert(0, SSP_PUTANJA)   # folder u kojem se nalazi paket ssp/
import ssp

import numpy as np
import matplotlib.pyplot as plt

# Podešavanje prikaza za rad studenata (preporuka iz README-a):
ssp.FORMATIRANJE_PRIKAZA = "JUPYTER"
ssp.REZOLUCIJA = "LOW"

N = 10000   # broj eksperimenata

In [ ]:
def gadjaj(sigma, broj_strelica):
    """Igrač sa preciznošću sigma baca strelice: greške po x i y su N(0, sigma^2)."""
    X = np.random.normal(0, sigma, broj_strelica)
    Y = np.random.normal(0, sigma, broj_strelica)
    return X, Y

def nacrtaj_metu(osa, poluprecnici):
    t = np.linspace(0, 2*np.pi, 200)
    for r in poluprecnici:
        osa.plot(r*np.cos(t), r*np.sin(t), "k", lw=0.8)
    osa.set_aspect("equal")

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for osa, sigma, ime in zip(ax, [2, 5], ["dobar igrač", "slabiji igrač"]):
    X, Y = gadjaj(sigma, 2000)
    nacrtaj_metu(osa, [2.5, 5, 10, 15, 20])
    osa.plot(X, Y, ".", ms=2)
    osa.set_xlim(-22, 22); osa.set_ylim(-22, 22)
    osa.set_title(f"{ime}: $\\sigma={sigma}$ (2000 strelica)")
fig.tight_layout(); plt.show()

## 10 000 strelica igrača sa $\sigma=5$ (kao $\eta=5$ u programskom kodu 2.3)
Primijeti: pogoci su najgušći u centru, ali je **najvjerovatnija udaljenost** $\sigma$, a ne 0.
Krug poluprečnika $r$ ima obim $2\pi r$, pa na većim udaljenostima ima „više mjesta“ za pogodak. Zato je $p(0)=0$.

In [ ]:
sigma = 5
X, Y = gadjaj(sigma, N)
R = np.sqrt(X**2 + Y**2)                          # udaljenost od centra mete

ssp.slprom1D.realizacije(np.arange(500), R[:500],
                         naslov="Udaljenost prvih 500 pogodaka od centra, $\\sigma=5$")

X_knjiga = np.random.rayleigh(sigma, N)          # programski kod 2.3
r = np.linspace(0, 5*sigma, 1000)
p_teor = r / sigma**2 * np.exp(-r**2 / (2*sigma**2))    # relacija (2.14)
P_teor = 1 - np.exp(-r**2 / (2*sigma**2))               # relacija (2.15)

xp, p_pikado = ssp.slprom1D.pdf(R, step=0.5, prosirenje_opsega=0.0, prikaz=False)
plt.figure(figsize=(6, 3.5))
plt.hist(R, bins=50, density=True, alpha=0.6, label="pikado (histogram)")
plt.plot(xp, p_pikado, "b.", ms=4, label="pikado (ssp.slprom1D.pdf)")
plt.hist(X_knjiga, bins=50, density=True, histtype="step", lw=1.5, color="k", label="np.random.rayleigh")
plt.plot(r, p_teor, "r", lw=2, label="teorija (2.14)")
plt.axvline(sigma, color="gray", ls=":", label="najvjerovatnija vrijednost $\\sigma$")
plt.title("Funkcija gustine vjerovatnoće $p(r)$"); plt.xlabel("udaljenost od centra $r$")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

xs, Px = ssp.slprom1D.PDF(R, step=0.1, prosirenje_opsega=0.1, prikaz=False)
ssp.slprom1D.grafik(xs, Px, r, P_teor, stil2="r--",
                    x_labela="$r$", naslov="Funkcija raspodjele vjerovatnoće $P(r)$",
                    legenda=["pikado", "teorija (2.15)"], lokacija_lgd="lower right")

print(f"E[R]   = {np.mean(R):.4f}  (teorija sigma*sqrt(pi/2)   = {sigma*np.sqrt(np.pi/2):.4f})")
print(f"Var[R] = {np.var(R):.4f}  (teorija (4-pi)/2*sigma^2 = {(4-np.pi)/2*sigma**2:.4f})")
print(f"medijan = {np.median(R):.4f}  (teorija sigma*sqrt(2 ln 2)  = {sigma*np.sqrt(2*np.log(2)):.4f})")

## Koliko često igrač pogodi krug poluprečnika $r$?
Iz (2.15) slijedi direktan odgovor: $P(R\le r)=1-e^{-r^2/(2\sigma^2)}$. Npr. u krug poluprečnika $\sigma$ ulazi
$1-e^{-1/2}\approx39{,}3\%$ strelica, a u krug $2\sigma$ ulazi $1-e^{-2}\approx86{,}5\%$.

In [ ]:
for k in [1, 2, 3]:
    print(f"P(R <= {k}*sigma) = {np.mean(R <= k*sigma):.4f}   (teorija {1 - np.exp(-k**2/2):.4f})")